# ARC-AGI-2: conceptual lattice + RDR over ARCGraph objects (symbolic, no neural model)
Frozen prior vocabulary `fill_interior,ray,connect,border,bbox_fill,outline8,L_inertia,D12,S_holes,D4,D1,S_size,P_pixel,P_frame,D10,S_atoms,G_dsl,L_lines,L_concepts,L_priors2,L_priors3`. Per task, RDR decision lists over lattice concepts are induced
from the training pairs only and executed on the test inputs. Unresolved slots get a 1x1 placeholder,
so any non-zero score comes from the induced rules.










**Probe:** V34 = V33 + 19 second-pass prior families (stratum L_priors3; fills empty answer slots only). Driver: round scheduler on the test run (deadline 11 h, 10 rounds, 25 % overrun skip, per-task commit); parity run unchanged. Logging (v22): prediction-set sha256 and per-task wall times are printed; predictions unchanged.


In [ ]:
import hashlib, json, subprocess, sys, zipfile, os, time
NB_T0 = time.time()   # round scheduler: the hidden-test run must finish by NB_T0 + DEADLINE_H hours
from pathlib import Path
BUNDLE_SHA = '9eedcc404d6015a2c84db58c63ec97c3e55f282faaed5fb9eb479afe0a6ab4fb'
bundles = [p for p in Path('/kaggle/input').rglob('candidate.bundle')]
assert len(bundles) == 1, bundles
payload = bundles[0].parent
assert hashlib.sha256(bundles[0].read_bytes()).hexdigest() == BUNDLE_SHA
cand = Path('/kaggle/working/cand')
if not cand.exists():
    with zipfile.ZipFile(bundles[0]) as z:
        z.extractall(cand)
deps = Path('/kaggle/working/deps')
try:
    import rdflib  # noqa
except ImportError:
    subprocess.run([sys.executable, '-m', 'pip', 'install', '--no-index', '--no-deps', '--target', str(deps),
                    str(payload / 'rdflib-7.6.0-py3-none-any.whl')], check=True)
challenges = [p for p in Path('/kaggle/input').rglob('arc-agi_test_challenges.json')]
assert len(challenges) == 1, challenges
Path('/kaggle/working/probe').mkdir(exist_ok=True)
CODEX_SHA = '62d4c71378e546093b874b9a1a1ffc61758b85cd531c06b5d4ecab33df18693b'
codex = [p for p in Path('/kaggle/input').rglob('codex_atoms.tgz')]
if codex:
    import tarfile
    assert hashlib.sha256(codex[0].read_bytes()).hexdigest() == CODEX_SHA, 'codex atom bundle digest mismatch'
    cx = Path('/kaggle/working/codex')
    if not cx.exists():
        with tarfile.open(codex[0]) as t: t.extractall(cx)
    os.environ['CODEX_ROOT'] = str(cx); os.environ['S0_REPS'] = str(cx / 's0_reps.json')
print(payload, challenges[0], 'codex' if codex else 'no-codex')


In [ ]:
# Probe payload (lzma + base64): 109 files (UTF-8 text), written byte-identical to /kaggle/working/probe
import base64 as _b64, hashlib as _hl, json as _json, lzma as _lzma, os as _os
_os.makedirs('/kaggle/working/probe', exist_ok=True)
_P = (
    '...'
)
_F = _json.loads(_lzma.decompress(_b64.b64decode(_P)))
for _name in sorted(_F):
    open(_os.path.join('/kaggle/working/probe', _name), 'wb').write(_F[_name].encode('utf-8'))
_set = _hl.sha256(''.join(_n + _hl.sha256(_F[_n].encode('utf-8')).hexdigest() for _n in sorted(_F)).encode()).hexdigest()
assert _set == '53e6f51221241a88e97dc7eca41b51022fc1d8855319a46e122d7246b8313ed0', 'probe payload digest mismatch'
print(len(_F), 'probe files written; set sha', _set[:16])


In [ ]:
%%writefile /kaggle/working/predict_m1.py
"""Kaggle driver for the M1b lattice/RDR probe (occupancy2.py), with the round scheduler (Fable G27-G29 / v17 §4).

Every prediction comes from rules induced on the task's own training pairs over the frozen prior
vocabulary; unresolved slots get a 1x1 [[0]] placeholder. Deterministic (sorted iteration), so a
local run and the Kaggle run over the same file must produce the same submission digest.

Round scheduler (only when --deadline-at is given; the parity run never passes it, so its predictions and digest are
unchanged):
  - the sorted tasks are cut into --rounds equal rounds; each round's budget is (deadline - start) / rounds
  - per-round commit: the whole submission (finished tasks + placeholders for the rest) is written atomically after
    every task, so a valid file always exists
  - 25 % overrun skip: once a round has used more than --overrun x its budget, its remaining tasks get placeholders
    and the next round starts on time
  - hard stop: no task starts within --reserve seconds of the deadline; each task's alarm is capped by the time left
A guard report (tasks run / skipped, rounds overrun, seconds) goes to <out>.guard.json.

Logging (v22, Fable v23 A.2; predictions unchanged): per-task wall seconds go to <out>.times.json, and the final
JSON line also carries the prediction-set sha256 ("digest", as before), the number of tasks over 120 s and the
slowest task's seconds.
"""
import argparse, hashlib, json, math, os, signal, sys, time

PLACEHOLDER = [[0]]


class TaskTimeout(BaseException):
    pass


def write_atomic(path, obj):
    tmp = path + '.tmp'
    with open(tmp, 'w') as f: f.write(json.dumps(obj))
    os.replace(tmp, path)


def main():
    ap = argparse.ArgumentParser()
    ap.add_argument("--candidate", required=True)      # dir containing image.py / ARCGraph.py
    ap.add_argument("--probe", required=True)          # dir containing occupancy2.py
    ap.add_argument("--vocab", required=True)
    ap.add_argument("--challenges", required=True)
    ap.add_argument("--out", required=True)
    ap.add_argument("--task-seconds", type=int, default=120)
    ap.add_argument("--deadline-at", type=float, default=0.0)   # absolute epoch seconds; 0 = no scheduler
    ap.add_argument("--rounds", type=int, default=10)
    ap.add_argument("--overrun", type=float, default=1.25)
    ap.add_argument("--reserve", type=float, default=120.0)
    args = ap.parse_args()
    os.environ["M1B_VOCAB"] = args.vocab
    sys.path[:0] = [args.probe, args.candidate]
    import occupancy2 as probe
    probe.Timeout = TaskTimeout
    ch = json.load(open(args.challenges))
    signal.signal(signal.SIGALRM, lambda *a: (_ for _ in ()).throw(TaskTimeout()))
    keys = sorted(ch)
    sub = {k: [{"attempt_1": PLACEHOLDER, "attempt_2": PLACEHOLDER} for _ in ch[k]["test"]] for k in keys}
    sched = args.deadline_at > 0
    t0 = time.time()
    per = max(1, math.ceil(len(keys) / max(1, args.rounds)))
    budget = (args.deadline_at - t0) / max(1, args.rounds) if sched else 0.0
    rep = {"tasks": len(keys), "run": 0, "skipped_overrun": 0, "skipped_deadline": 0, "rounds_overrun": [], "scheduler": sched}
    round_start, cur = t0, -1
    times = {}
    for n, k in enumerate(keys):
        task = ch[k]
        if sched:
            r = n // per
            if r != cur:
                cur, round_start = r, time.time()
            now = time.time()
            if now + args.reserve > args.deadline_at:
                rep["skipped_deadline"] += 1; continue
            if now - round_start > args.overrun * budget:
                if not rep["rounds_overrun"] or rep["rounds_overrun"][-1] != r: rep["rounds_overrun"].append(r)
                rep["skipped_overrun"] += 1; continue
            secs = int(max(1, min(args.task_seconds, args.deadline_at - args.reserve - now)))
        else:
            secs = args.task_seconds
        slots = sub[k]
        t_task = time.time()
        try:
            signal.alarm(secs)
            attempts, _, _ = probe.solve(task)
            for rank, a in enumerate(attempts[:2]):
                for i, g in enumerate(a["preds"]):
                    if g and all(isinstance(r_, list) and r_ for r_ in g):
                        slots[i]["attempt_1" if rank == 0 else "attempt_2"] = g
        except TaskTimeout:
            pass
        except Exception:
            pass
        finally:
            signal.alarm(0)
            times[k] = round(time.time() - t_task, 2)
        rep["run"] += 1
        if sched: write_atomic(args.out, sub)
    blob = json.dumps(sub, sort_keys=True).encode()
    open(args.out, "w").write(json.dumps(sub))
    rep["seconds"] = round(time.time() - t0, 1)
    if sched: json.dump(rep, open(args.out + ".guard.json", "w"), indent=1)
    json.dump(times, open(args.out + ".times.json", "w"), indent=0, sort_keys=True)
    tv = list(times.values())
    log = {"tasks_timed": len(tv), "max_task_s": max(tv) if tv else 0.0, "n_over_120s": sum(t > 120 for t in tv),
           "total_task_s": round(sum(tv), 1)}
    print(json.dumps({"tasks": len(sub), "digest": hashlib.sha256(blob).hexdigest(), **log, **({"guard": rep} if sched else {})}))


if __name__ == "__main__":
    main()


In [ ]:
# Parity gate: the notebook must replicate the local result on the public 120-task evaluation set.
PROBE_SHA = 'b9f8968f8f5154d4ea7241f5d9c1915ab7e7a082b1d5376e278354625e7b2339'
VOCAB = 'fill_interior,ray,connect,border,bbox_fill,outline8,L_inertia,D12,S_holes,D4,D1,S_size,P_pixel,P_frame,D10,S_atoms,G_dsl,L_lines,L_concepts,L_priors2,L_priors3'
EVAL_SHA = 'e7c62a4bd211867c6b538f66b8013b81f299663c82ca062f49a52bf439d6e4e8'
EXPECTED_DIGEST = 'b6c6bfd8fd763162973b4c730ba82cfc1aaa3728f83fec7bffb060b3638b4b49'
assert hashlib.sha256(Path('/kaggle/working/probe/occupancy2.py').read_bytes()).hexdigest() == PROBE_SHA
evals = [p for p in Path('/kaggle/input').rglob('arc-agi_evaluation_challenges.json')
         if hashlib.sha256(p.read_bytes()).hexdigest() == EVAL_SHA]
assert evals, 'identical public evaluation file not found; parity cannot be verified'
sols = list(evals[0].parent.glob('arc-agi_evaluation_solutions.json'))
env = dict(os.environ, PYTHONHASHSEED='0', OMP_NUM_THREADS='1', MPLBACKEND='Agg',
           PYTHONPATH=os.pathsep.join([str(deps), os.environ.get('PYTHONPATH', '')]))
def run(ch, out):
    subprocess.run([sys.executable, '/kaggle/working/predict_m1.py', '--candidate', str(cand),
                    '--probe', '/kaggle/working/probe', '--vocab', VOCAB, '--challenges', str(ch),
                    '--out', out, '--task-seconds', '300'], check=True, env=env)
    return json.load(open(out))
ev = run(evals[0], '/kaggle/working/parity_eval.json')
digest = hashlib.sha256(json.dumps(ev, sort_keys=True).encode()).hexdigest()
report = {'eval_file': str(evals[0]), 'prediction_digest': digest, 'digest_match': digest == EXPECTED_DIGEST,
          'correct_count': 'not computed (G18: includes sealed tasks)'}
json.dump(report, open('/kaggle/working/parity_report.json', 'w'), indent=1)
print(report)
assert digest == EXPECTED_DIGEST, 'PARITY FAILED: predictions differ from the frozen local run'


In [ ]:
# Round scheduler (Fable G27-G29): the test run gets an absolute deadline; tasks it cannot reach get placeholders.
DEADLINE_H = 11.0
subprocess.run([sys.executable, '/kaggle/working/predict_m1.py', '--candidate', str(cand),
                '--probe', '/kaggle/working/probe', '--vocab', VOCAB, '--challenges', str(challenges[0]),
                '--out', '/kaggle/working/submission.json', '--task-seconds', '300',
                '--deadline-at', str(NB_T0 + DEADLINE_H * 3600), '--rounds', '10', '--overrun', '1.25'],
               check=True, env=env)
sub = json.load(open('/kaggle/working/submission.json'))
src = json.load(open(challenges[0]))
assert set(sub) == set(src) and all(len(sub[k]) == len(src[k]['test']) for k in src)
print(len(sub), 'tasks written;', open('/kaggle/working/submission.json.guard.json').read())
# Log line (v22, Fable v23 A.2): sha256 of the prediction set and per-task wall times; predictions unchanged.
_times = json.load(open('/kaggle/working/submission.json.times.json'))
print('PREDICTION_SET_SHA256', hashlib.sha256(json.dumps(sub, sort_keys=True).encode()).hexdigest())
print('TASK_TIMES', json.dumps({'n': len(_times), 'max_s': max(_times.values()) if _times else 0.0, 'n_over_120s': sum(v > 120 for v in _times.values()), 'total_s': round(sum(_times.values()), 1)}))
print('TASK_TIMES_SORTED', json.dumps(sorted(_times.values(), reverse=True)))   # anonymous: no task ids are printed
